In [1]:
import pandas as pd
import sqlite3 as db

In [5]:
import pandas as pd

# ── 1. Population data (your existing code, unchanged) ──────────────────────
pop = pd.read_csv("county_estimates.csv", sep=";")
pop.columns = (
    pop.columns
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)
pop["FIPS"] = pop["FIPS"].astype(str).str.zfill(5)
pop["Year"] = pd.to_datetime(pop["Year"], errors="coerce").dt.year

pop = pop[
    (pop["Area Type"] == "County") &
    (pop["Estimate"] == "Annual Estimate")
][["FIPS", "Area", "Year", "value"]].rename(columns={"value": "population"})

# ── 2. Crash data (your existing code, unchanged) ───────────────────────────
try:
    tr = pd.read_csv("nc_transportation_line_all_data.csv")
except UnicodeDecodeError:
    tr = pd.read_csv("nc_transportation_line_all_data.csv", encoding="utf-16")
tr["Geo_ID"] = tr["Geo_ID"].astype(str).str.zfill(5)

crashes = tr[
    (tr["Variable"] == "Traffic Accidents")
][["Geo_ID", "Area_Name", "Year", "Value"]].rename(
    columns={"Geo_ID": "FIPS", "Value": "crashes"}
)

# ── 3. NEW: Load and pivot the LINC formatted file ──────────────────────────
linc = pd.read_csv("nc-transportation-linc-formatted.csv")
linc["Geo_ID"] = linc["Geo_ID"].astype(str).str.zfill(5)

# Filter to Mecklenburg only, then pivot long → wide
linc_meck = linc[linc["Geo_ID"] == "37119"].copy()

linc_wide = linc_meck.pivot_table(
    index=["Geo_ID", "Year"],
    columns="Variable",
    values="Value",
    aggfunc="first"
).reset_index().rename(columns={"Geo_ID": "FIPS"})

# Flatten column names (removes the "Variable" label from the MultiIndex)
linc_wide.columns.name = None

# ── 4. Merge everything together ─────────────────────────────────────────────
merged = crashes.merge(pop, on=["FIPS", "Year"], how="inner")
merged = merged.merge(linc_wide, on=["FIPS", "Year"], how="inner")

# ── 5. Filter to Mecklenburg, deduplicate, sort (your existing logic) ────────
df = merged[merged["FIPS"] == "37119"].copy()
df = df.sort_values("Year").drop_duplicates(subset=["Year"])

# ── 6. Growth variables (your existing code, unchanged) ──────────────────────
df["pop_change"] = df["population"].diff()
df["pop_growth_pct"] = df["population"].pct_change() * 100
df["crash_growth_pct"] = df["crashes"].pct_change() * 100

# ── 7. Per-capita normalization of key LINC features ────────────────────────
per_capita_vars = [
    "Traffic Accident Fatalities",
    "Alcohol Related Accidents",
    "Accidents Caused by Exceeding Legal Speed",
    "Motorcyclists Injured",
    "Motorcyclist Fatalities",
    "Pedestrian Accidents",
    "Pedestrians Injured",
    "Bicyclists Injured",
]
for var in per_capita_vars:
    if var in df.columns:
        df[f"{var}_per_capita"] = df[var] / df["population"] * 100_000

# ── 8. Create binary target label ────────────────────────────────────────────
median_fatalities = df["Traffic Accident Fatalities"].median()
df["high_fatality"] = (df["Traffic Accident Fatalities"] > median_fatalities).astype(int)

print(f"Median fatalities: {median_fatalities}")
print(f"Label distribution:\n{df['high_fatality'].value_counts()}")
print(f"\nFinal dataset shape: {df.shape}")
print(f"Year range: {df['Year'].min()} – {df['Year'].max()}")

Median fatalities: 74.0
Label distribution:
high_fatality
1    22
0    22
Name: count, dtype: int64

Final dataset shape: (44, 38)
Year range: 1975 – 2024
